# Step 9 — Latency Measurement (Table 6)

Measures the two heavy pipeline components for real, 100 runs each, on this machine:
- **PANNs CNN14 inference** (on a real ESC-50 clip, the same preprocessing as ESAS)
- **GCC-PHAT localization** (6-pair median on a 1.5 s, 48 kHz window)

Audio buffering is a design constant (the capture buffer length), not a measurement. The alert-engine time can only be measured from your live ESAS runtime script — if you have one, tell Claude; otherwise Table 6 will be restructured around what is actually measured.

**Output:** `results/latency.json` + a printed summary to paste back.

In [ ]:
from pathlib import Path

# ── Base path Set your base path here ─────────────────
BASE = Path('/path/to/your/esas_project')  # <-- set this
# ────────────────────────────────────────────────────────────
ESC50_DIR   = BASE / 'ESC-50'
PANNS_CKPT  = BASE / 'panns_data' / 'Cnn14_mAP=0.431.pth'
RESULTS_DIR = Path('results'); RESULTS_DIR.mkdir(exist_ok=True)
print(f'ESC-50: {ESC50_DIR.exists()}   PANNs ckpt: {PANNS_CKPT.exists()}')

In [ ]:
import glob, json, math, time
import numpy as np
import librosa
from panns_inference import AudioTagging

panns = AudioTagging(checkpoint_path=str(PANNS_CKPT), device='cpu')
wav = sorted(glob.glob(str(ESC50_DIR / 'audio' / '*.wav')))[0]
audio, _ = librosa.load(wav, sr=32000, mono=True)
audio = audio[:64000] if len(audio) >= 64000 else np.pad(audio, (0, 64000-len(audio)))
clip = audio[None, :]

panns.inference(clip)  # warm-up, not timed
times = []
for _ in range(100):
    t0 = time.perf_counter()
    panns.inference(clip)
    times.append((time.perf_counter() - t0) * 1000)
panns_ms = np.array(times)
print(f'PANNs CNN14 inference: mean={panns_ms.mean():.0f} ms   P95={np.percentile(panns_ms,95):.0f} ms   (n=100)')

In [ ]:
SR, D, C = 48_000, 0.060, 343.0
def gcc_phat(s1, s2):
    n   = 2*int(2**math.ceil(math.log2(max(len(s1),len(s2)))))
    X1  = np.fft.rfft(s1, n=n); X2 = np.fft.rfft(s2, n=n)
    cc  = X1*np.conj(X2)
    gcc = np.fft.irfft(cc/(np.abs(cc)+1e-10), n=n)
    ml  = int(SR*D/C)
    gh  = np.concatenate([gcc[-ml:], gcc[:ml+1]])
    pk  = int(np.argmax(gh)) - ml
    return math.degrees(math.asin(np.clip(pk/SR*C/D,-1,1)))

rng = np.random.default_rng(0)
win = rng.standard_normal((7, int(SR*1.5))).astype(np.float32)
PAIRS = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
float(np.median([gcc_phat(win[a], win[b]) for a,b in PAIRS]))  # warm-up
times = []
for _ in range(100):
    t0 = time.perf_counter()
    float(np.median([gcc_phat(win[a], win[b]) for a,b in PAIRS]))
    times.append((time.perf_counter() - t0) * 1000)
gcc_ms = np.array(times)
print(f'GCC-PHAT (6-pair median, 1.5 s window): mean={gcc_ms.mean():.0f} ms   P95={np.percentile(gcc_ms,95):.0f} ms   (n=100)')

In [ ]:
BUFFER_MS = 80  # design constant: capture buffer length
summary = {
    'audio_buffering_ms': BUFFER_MS,
    'panns_mean_ms': round(float(panns_ms.mean())), 'panns_p95_ms': round(float(np.percentile(panns_ms,95))),
    'gcc_mean_ms':   round(float(gcc_ms.mean())),   'gcc_p95_ms':   round(float(np.percentile(gcc_ms,95))),
    'measured_total_mean_ms': BUFFER_MS + round(float(panns_ms.mean())) + round(float(gcc_ms.mean())),
    'measured_total_p95_ms':  BUFFER_MS + round(float(np.percentile(panns_ms,95))) + round(float(np.percentile(gcc_ms,95))),
    'note': 'alert engine not measured (no runtime script provided); totals exclude it'
}
print('='*56)
print('  LATENCY SUMMARY (paste this block back to Claude)')
print('='*56)
for k, v in summary.items(): print(f'  {k}: {v}')
print('='*56)
json.dump(summary, open(RESULTS_DIR / 'latency.json', 'w'), indent=2)
print(f'Saved: {RESULTS_DIR}/latency.json')